# E6R analysis consumer — the depth ablation, reps 1 versus reps 2

E6 measured the flat-encoded reps-1 representation on four fixed-degree-
sequence strata and found the collapse; E6D measured the degree-encoded
reps-1 arm, and degree encoding did not restore decodability, so both
encoder arms are measured negatives and repetition is the remaining
circuit knob. E6R re-embedded the identical 1,600 graphs with the flat
encoder and the (entangle, mix) block applied twice. This consumer
completes the depth comparison: same graphs, same targets, same frozen
splits, same probes, reps as the only difference. The flat reps-1 results
are loaded from `n14_e6_analysis.pkl` and are not recomputed or altered;
the degree-encoded reps-2 arm is deferred per Luke.

**Pre-registration.** No directional expectation is registered. The
second (entangle, mix) block acts on the mixed state, so the composition
is genuinely deeper interference rather than a doubled angle, and whether
depth restores any off-regularity decodability is the open question.
Three sub-questions are registered so the table is read against them
rather than post hoc:
1. Does depth lift the QuIC column on any cycle row, with C5 as the live
   row exactly as in E6?
2. Does the flat run's max-degree gradient (healthiest in the Δ = 4
   strata, dead in Δ = 5 and Δ = 6) persist, invert, or vanish under
   depth?
3. Do the flat run's only positive cells (dConcat on C5 in S1/S2,
   +0.061 and +0.118) persist, grow, or vanish?

**Consistency checks built into the comparison.** The eig and moments
columns are functions of the graphs alone, and the graphs are asserted
identical, so this notebook recomputes both columns and asserts them
equal to the flat run's values fold by fold. The moments column must
again sit at exactly 1.000 on C3 and C4 (identities), and the eig column
at 1.000 on spectral gap. One graph per stratum is recomputed through
the flat-encoder reps-2 qiskit circuit and asserted against the stored
vector at 1e-12, which catches pointing this consumer at the wrong pkl.

**Protocol, identical to the flat consumer.** RidgeCV(alphas =
logspace(-14, 2, 17), cv = 5) per fold; KFold(5, shuffle, seed 0) splits
regenerated from seed and asserted equal to the splits saved with the
flat analysis; spectral blocks standardized in-fold; QuIC block raw;
concat rescales the eigenvalue block per train fold to match the QuIC
block's RMS. Degenerate targets and constant-fold guards carry over
unchanged.

**Runtime.** The quic and concat ridges on the reps-2 vectors dominate,
about 2 to 4 hours for the full grid, matching the flat run. RUN_STRATA
is the session knob; results checkpoint per stratum to
`n14_e6r_analysis.pkl`.

## Environment

In [1]:
# qiskit is required for the spot check, which recomputes one stored vector
# per stratum through the verbatim census pipeline.
!pip install 'qiskit[visualization]' --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 90.3 MB/s eta 0:00:00


In [2]:
import pickle

import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from tqdm.notebook import tqdm

from scipy.linalg import LinAlgWarning
import warnings
warnings.filterwarnings('ignore', category=LinAlgWarning)

In [3]:
SEED = 0
RIDGE_ALPHAS = np.logspace(-14, 2, 17)
NUM_FOLDS = 5

# Inputs: the degree-encoded dataset, the flat dataset, and the flat analysis.
DEPTH_E6R_BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14-e6r-dataset-producer/"
FLAT_E6_BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14-e6-dataset-producer/"
FLAT_ANALYSIS_BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14-e6-analysis-consumer/"

NUM_VERTICES = 14
EXPECTED_STRATA = ('S1_near_regular', 'S2_bimodal', 'S3_skewed', 'S4_hub')
EXPECTED_GRAPHS_PER_STRATUM = 400
TARGET_NAMES = ('C3', 'C4', 'C5', 'C6', 'girth', 'diameter',
                'radius', 'wiener', 'spectral_gap')
MOMENT_POWERS = (3, 4, 5, 6, 7, 8)

# Circuit parameters — canonical angles, reps=2 (spot check only).
FLAT_ENCODING_ANGLE = 2.875
ENTANGLER_SCALAR = 2.0
MIXER_SCALAR = 0.1
REPS = 2

SPOT_CHECK_TOLERANCE = 1e-12       # stored vector vs qiskit recompute
SHARED_COLUMN_TOLERANCE = 1e-6     # recomputed eig/moments vs flat run's values

# Session knob: subset of strata for parallel Kaggle launches.
RUN_STRATA = EXPECTED_STRATA

## Load all three inputs and assert graph identity

The degree pkl must carry the identical graphs as the flat pkl (asserted
string for string on canonical graph6), identical copied targets, and a
meta record saying degree encoding. The flat analysis pkl supplies the
frozen splits and the flat columns.

In [4]:
with open(DEPTH_E6R_BASE + "n14_e6r_data_dict.pkl", 'rb') as depth_file:
    depth_pkl = pickle.load(depth_file)
with open(FLAT_E6_BASE + "n14_e6_data_dict.pkl", 'rb') as flat_file:
    flat_pkl = pickle.load(flat_file)
with open(FLAT_ANALYSIS_BASE + "n14_e6_analysis.pkl", 'rb') as analysis_file:
    flat_analysis = pickle.load(analysis_file)

assert depth_pkl['meta']['circuit']['encoding'].startswith('flat'), (
    'depth pkl is not flat-encoded')
assert depth_pkl['meta']['circuit']['reps'] == 2, (
    'depth pkl is not the reps-2 E6R run')
assert flat_pkl['meta']['circuit']['encoding'].startswith('flat'), (
    'flat pkl is not the flat-encoder E6 run')
assert flat_pkl['meta']['circuit']['reps'] == 1, (
    'flat pkl is not the reps-1 run')
assert tuple(sorted(depth_pkl['data'])) == tuple(sorted(EXPECTED_STRATA))
assert tuple(sorted(flat_analysis['results'])) == tuple(sorted(EXPECTED_STRATA)), (
    'flat analysis pkl does not cover all four strata')

STRATA = {}
for stratum_name in EXPECTED_STRATA:
    depth_entries = depth_pkl['data'][stratum_name]
    flat_entries = flat_pkl['data'][stratum_name]
    graph_indices = sorted(depth_entries)
    assert len(graph_indices) == EXPECTED_GRAPHS_PER_STRATUM
    assert graph_indices == sorted(flat_entries)

    for graph_index in graph_indices:
        # graph identity: the ablation is void if these ever differ
        assert (depth_entries[graph_index]['graph6']
                == flat_entries[graph_index]['graph6']), (
            f'{stratum_name} graph {graph_index}: graph6 mismatch between arms')

    embedding_matrix = np.vstack(
        [depth_entries[graph_index]['exact_vector']
         for graph_index in graph_indices])
    assert embedding_matrix.shape == (len(graph_indices), 1 << NUM_VERTICES)
    assert np.all(np.diff(embedding_matrix, axis=1) <= 0), (
        f'{stratum_name}: reps-2 vectors not sorted descending')
    assert np.abs(embedding_matrix.sum(axis=1) - 1.0).max() < 1e-12, (
        f'{stratum_name}: reps-2 vectors not normalized')

    STRATA[stratum_name] = {
        'graph_indices': graph_indices,
        'adjacency_matrices': np.stack(
            [depth_entries[graph_index]['adj_mat']
             for graph_index in graph_indices]).astype(np.int64),
        'embedding_matrix': embedding_matrix,
        'eigenvalue_matrix': np.vstack(
            [depth_entries[graph_index]['spectrum']
             for graph_index in graph_indices]),
        'target_values': {target_name: np.array(
            [depth_entries[graph_index][target_name]
             for graph_index in graph_indices], dtype=float)
            for target_name in TARGET_NAMES},
    }
    print(f'{stratum_name}: {len(graph_indices)} graphs, graph identity with '
          f'flat arm asserted, reps-2 vectors verified')

S1_near_regular: 400 graphs, graph identity with flat arm asserted, reps-2 vectors verified
S2_bimodal: 400 graphs, graph identity with flat arm asserted, reps-2 vectors verified
S3_skewed: 400 graphs, graph identity with flat arm asserted, reps-2 vectors verified
S4_hub: 400 graphs, graph identity with flat arm asserted, reps-2 vectors verified


## Spot check through the qiskit pipeline

The class below is the E6 flat-encoder variant of the census
`QuIK_circuit`, constructed at reps = 2 exactly as in the E6R producer.
One graph per stratum is rebuilt and simulated through qiskit
`Statevector`; the sorted probabilities must match the stored vector at
1e-12. A failure means this consumer is pointed at the wrong pkl.

In [5]:
class QuIK_circuit:
    """Builds the QuIC circuit for a given adjacency matrix.

    Flat RX encoder (the E6 variant), RZZ entangler over edges, uniform RX
    mixer, (entangler + mixer) repeated `reps` times. Construction only.

    Census class with the E6 flat-encoder line; reps = 2 is the E6R knob.
    """

    def __init__(self, adj_mat,
                 flat_encoding_angle=FLAT_ENCODING_ANGLE,
                 entangler_scalar=ENTANGLER_SCALAR,
                 mixer_scalar=MIXER_SCALAR,
                 reps=REPS):
        self.adj_mat = np.asarray(adj_mat)
        self.flat_encoding_angle = flat_encoding_angle
        self.entangler_scalar = entangler_scalar
        self.mixer_scalar = mixer_scalar
        self.reps = reps

        self.num_qubits = self.adj_mat.shape[0]
        self.qubits = list(range(self.num_qubits))

        self.quik_circuit = self.build_quik_circuit()

    def build_encoder(self):
        encoder = QuantumCircuit(self.num_qubits)
        encoder.rx(self.flat_encoding_angle, self.qubits, 'Flat Encoder')
        return encoder

    def build_entangler(self):
        entangler = QuantumCircuit(self.num_qubits)
        edge_u, edge_v = np.nonzero(np.triu(self.adj_mat, k=1))
        for u, v in zip(edge_u, edge_v):
            entangler.rzz(self.entangler_scalar, u, v)
        return entangler

    def build_mixer(self):
        mixer = QuantumCircuit(self.num_qubits)
        mixer.rx(self.mixer_scalar, self.qubits, 'Mixer')
        return mixer

    def build_quik_circuit(self):
        self.encoder = self.build_encoder()
        self.mixer = self.build_mixer()
        self.entangler = self.build_entangler()

        circuit = self.encoder.copy()
        for _ in range(self.reps):
            circuit.append(self.entangler, self.qubits)
            circuit.append(self.mixer, self.qubits)
        circuit.measure_all()
        return circuit


for stratum_name in EXPECTED_STRATA:
    first_adjacency = STRATA[stratum_name]['adjacency_matrices'][0]
    spot_circuit = QuIK_circuit(first_adjacency).quik_circuit.decompose()
    spot_circuit = spot_circuit.remove_final_measurements(inplace=False)
    spot_probabilities = Statevector.from_instruction(spot_circuit).probabilities()
    spot_vector = np.sort(spot_probabilities)[::-1]
    stored_vector = STRATA[stratum_name]['embedding_matrix'][0]
    max_coordinate_deviation = np.abs(spot_vector - stored_vector).max()
    assert max_coordinate_deviation < SPOT_CHECK_TOLERANCE, (
        f'{stratum_name}: spot check FAILED at {max_coordinate_deviation:.2e} '
        f'— wrong pkl or wrong encoder')
    print(f'{stratum_name}: qiskit spot check passed '
          f'({max_coordinate_deviation:.2e})')

S1_near_regular: qiskit spot check passed (0.00e+00)
S2_bimodal: qiskit spot check passed (0.00e+00)
S3_skewed: qiskit spot check passed (0.00e+00)
S4_hub: qiskit spot check passed (0.00e+00)


## Frozen splits — regenerated and asserted against the flat run

The splits are regenerated from seed 0 and asserted fold for fold against
the splits the flat analysis saved. Identical splits are what make the
per-fold comparison exact.

In [6]:
FOLD_SPLITS = {}
for stratum_name in EXPECTED_STRATA:
    fold_generator = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    FOLD_SPLITS[stratum_name] = list(
        fold_generator.split(np.arange(EXPECTED_GRAPHS_PER_STRATUM)))

    saved_splits = flat_analysis['splits'][stratum_name]
    for fold_number, (train_indices, test_indices) in enumerate(
            FOLD_SPLITS[stratum_name]):
        saved_train, saved_test = saved_splits[fold_number]
        assert train_indices.tolist() == saved_train, (
            f'{stratum_name} fold {fold_number}: train split mismatch with flat run')
        assert test_indices.tolist() == saved_test, (
            f'{stratum_name} fold {fold_number}: test split mismatch with flat run')
print('splits regenerated from seed 0 and asserted equal to the flat run, '
      'all strata, all folds')

splits regenerated from seed 0 and asserted equal to the flat run, all strata, all folds


## Probe machinery — identical to the flat consumer

The per-fold RidgeCV probe, the constant-fold guards, and the concat
construction (eigenvalue block rescaled per train fold to match the QuIC
block's root-mean-square) carry over unchanged. Variable names are
descriptive; the logic is line-for-line the flat consumer's.

In [7]:
def ridge_probe_scores(feature_matrix, target_values, fold_splits,
                       standardize=False):
    """Per-fold RidgeCV R^2 on the frozen splits (the E2 protocol).

    A fold whose train or test targets are constant records NaN rather
    than an ill-defined score; callers aggregate with NaN-aware means.
    Returns (fold_scores, chosen_alphas).
    """
    fold_scores, chosen_alphas = [], []
    for train_indices, test_indices in fold_splits:
        if (target_values[test_indices].std() == 0
                or target_values[train_indices].std() == 0):
            fold_scores.append(np.nan)
            chosen_alphas.append(np.nan)
            continue
        if standardize:
            probe_model = make_pipeline(StandardScaler(),
                                        RidgeCV(alphas=RIDGE_ALPHAS, cv=5))
        else:
            probe_model = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
        probe_model.fit(feature_matrix[train_indices],
                        target_values[train_indices])
        predictions = probe_model.predict(feature_matrix[test_indices])
        fold_scores.append(r2_score(target_values[test_indices], predictions))
        chosen_alphas.append(probe_model[-1].alpha_ if standardize
                             else probe_model.alpha_)
    return np.array(fold_scores), chosen_alphas


def concat_probe_scores(embedding_matrix, spectral_matrix, target_values,
                        fold_splits):
    """QuIC block raw + spectral block rescaled per train fold to match the
    QuIC block's root-mean-square, then one RidgeCV per fold."""
    fold_scores, chosen_alphas = [], []
    for train_indices, test_indices in fold_splits:
        if (target_values[test_indices].std() == 0
                or target_values[train_indices].std() == 0):
            fold_scores.append(np.nan)
            chosen_alphas.append(np.nan)
            continue
        embedding_rms = np.sqrt((embedding_matrix[train_indices] ** 2).mean())
        spectral_rms = np.sqrt((spectral_matrix[train_indices] ** 2).mean())
        spectral_scale = embedding_rms / spectral_rms
        train_features = np.hstack([embedding_matrix[train_indices],
                                    spectral_matrix[train_indices] * spectral_scale])
        test_features = np.hstack([embedding_matrix[test_indices],
                                   spectral_matrix[test_indices] * spectral_scale])
        probe_model = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
        probe_model.fit(train_features, target_values[train_indices])
        fold_scores.append(r2_score(target_values[test_indices],
                                    probe_model.predict(test_features)))
        chosen_alphas.append(probe_model.alpha_)
    return np.array(fold_scores), chosen_alphas

## Main grid

Per stratum: recompute the eig and moments columns and assert them equal
to the flat run's fold scores (the shared-column consistency check), then
run the quic and concat columns on the reps-2 vectors. Degenerate
targets mirror the flat run's flags by construction (identical graphs).
Results checkpoint per stratum.

In [8]:
RESULTS = {}
for stratum_name in RUN_STRATA:
    adjacency_matrices = STRATA[stratum_name]['adjacency_matrices']
    embedding_matrix = STRATA[stratum_name]['embedding_matrix']
    eigenvalue_matrix = STRATA[stratum_name]['eigenvalue_matrix']
    num_graphs = len(adjacency_matrices)

    # trace-moment block tr(A^k), k = 3..8 (k = 1 is zero, k = 2 is 2m,
    # both stratum constants)
    moment_matrix = np.empty((num_graphs, len(MOMENT_POWERS)))
    matrix_power = np.stack([np.linalg.matrix_power(adjacency, MOMENT_POWERS[0])
                             for adjacency in adjacency_matrices])
    moment_matrix[:, 0] = np.trace(matrix_power, axis1=1, axis2=2)
    for column_position, power in enumerate(MOMENT_POWERS[1:], start=1):
        matrix_power = matrix_power @ adjacency_matrices
        moment_matrix[:, column_position] = np.trace(matrix_power,
                                                     axis1=1, axis2=2)

    RESULTS[stratum_name] = {}
    for target_name in tqdm(TARGET_NAMES, desc=f'{stratum_name} targets'):
        target_values = STRATA[stratum_name]['target_values'][target_name]
        flat_row = flat_analysis['results'][stratum_name][target_name]

        if np.unique(target_values).size < 2:
            assert flat_row['degenerate'], (
                f'{stratum_name} {target_name}: degenerate here but not in the '
                f'flat run — graphs cannot be identical')
            RESULTS[stratum_name][target_name] = {'degenerate': True}
            print(f'{stratum_name} {target_name}: DEGENERATE — skipped, '
                  f'matching the flat run')
            continue
        assert not flat_row['degenerate'], (
            f'{stratum_name} {target_name}: non-degenerate here but degenerate '
            f'in the flat run — graphs cannot be identical')

        result_row = {'degenerate': False}

        # shared columns: recompute, then assert equality with the flat run
        for column_name, feature_matrix in (('eig', eigenvalue_matrix),
                                            ('moments', moment_matrix)):
            fold_scores, chosen_alphas = ridge_probe_scores(
                feature_matrix, target_values, FOLD_SPLITS[stratum_name],
                standardize=True)
            flat_fold_scores = np.asarray(flat_row[column_name]['r2_folds'])
            # Fold-guard NaNs must occur in the same folds in both runs
            # (identical graphs imply identical constant-fold patterns);
            # the numeric comparison then runs over the common finite folds.
            assert np.array_equal(np.isnan(fold_scores),
                                  np.isnan(flat_fold_scores)), (
                f'{stratum_name} {target_name} {column_name}: fold-guard NaN '
                f'pattern differs from the flat run — graphs cannot be identical')
            finite_fold_mask = ~np.isnan(fold_scores)
            if finite_fold_mask.any():
                max_fold_deviation = np.abs(
                    fold_scores[finite_fold_mask]
                    - flat_fold_scores[finite_fold_mask]).max()
            else:
                max_fold_deviation = 0.0
            assert max_fold_deviation < SHARED_COLUMN_TOLERANCE, (
                f'{stratum_name} {target_name} {column_name}: recomputed fold '
                f'scores deviate from the flat run by {max_fold_deviation:.2e} '
                f'— environment drift or graph mismatch; investigate before '
                f'trusting the comparison')
            result_row[column_name] = {
                'r2_folds': fold_scores,
                'r2_mean': float(np.nanmean(fold_scores)),
                'r2_sd': float(np.nanstd(fold_scores)),
                'n_nan_folds': int(np.isnan(fold_scores).sum()),
                'alphas_chosen': chosen_alphas,
                'max_deviation_vs_flat': float(max_fold_deviation)}

        # reps-2 columns
        for column_name, probe_call in (
                ('quic', lambda: ridge_probe_scores(
                    embedding_matrix, target_values, FOLD_SPLITS[stratum_name])),
                ('concat', lambda: concat_probe_scores(
                    embedding_matrix, eigenvalue_matrix, target_values,
                    FOLD_SPLITS[stratum_name]))):
            fold_scores, chosen_alphas = probe_call()
            result_row[column_name] = {
                'r2_folds': fold_scores,
                'r2_mean': float(np.nanmean(fold_scores)),
                'r2_sd': float(np.nanstd(fold_scores)),
                'n_nan_folds': int(np.isnan(fold_scores).sum()),
                'alphas_chosen': chosen_alphas}

        best_spectral_mean = float(np.nanmax([result_row['eig']['r2_mean'],
                                              result_row['moments']['r2_mean']]))
        result_row['best_spectral'] = best_spectral_mean
        result_row['delta_quic'] = (result_row['quic']['r2_mean']
                                    - best_spectral_mean)
        result_row['delta_concat'] = (result_row['concat']['r2_mean']
                                      - best_spectral_mean)
        RESULTS[stratum_name][target_name] = result_row

    with open('/kaggle/working/n14_e6r_analysis.pkl', 'wb') as output_file:
        pickle.dump({'results': RESULTS,
                     'splits': flat_analysis['splits'],
                     'seed': SEED, 'alphas_grid': RIDGE_ALPHAS,
                     'moment_powers': MOMENT_POWERS,
                     'run_strata': RUN_STRATA,
                     'flat_analysis_source': FLAT_ANALYSIS_BASE}, output_file)
    print(f'{stratum_name} complete — checkpointed to n14_e6r_analysis.pkl')

S1_near_regular targets:   0%|          | 0/9 [00:00<?, ?it/s]

S1_near_regular complete — checkpointed to n14_e6r_analysis.pkl


S2_bimodal targets:   0%|          | 0/9 [00:00<?, ?it/s]

S2_bimodal complete — checkpointed to n14_e6r_analysis.pkl


S3_skewed targets:   0%|          | 0/9 [00:00<?, ?it/s]

S3_skewed complete — checkpointed to n14_e6r_analysis.pkl


S4_hub targets:   0%|          | 0/9 [00:00<?, ?it/s]

S4_hub complete — checkpointed to n14_e6r_analysis.pkl


## Comparison tables — reps 1 versus reps 2, per stratum and aggregate

The eig and moments columns are shared (asserted equal above), so the
tables print them once and place the two depth arms side by side on the
quic and concat columns. Delta columns are reps-2 minus reps-1.

In [9]:
for stratum_name in RUN_STRATA:
    print(f'\n=== {stratum_name} — reps 1 vs reps 2 (flat encoder) ===')
    print(f'{"target":>13} | {"best_spec":>9} | {"quic_r1":>13} '
          f'{"quic_r2":>13} {"d(r2-r1)":>11} | {"cat_r1":>13} '
          f'{"cat_r2":>13} {"d(r2-r1)":>11}')
    for target_name in TARGET_NAMES:
        depth_row = RESULTS[stratum_name][target_name]
        flat_row = flat_analysis['results'][stratum_name][target_name]
        if depth_row['degenerate']:
            print(f'{target_name:>13} | degenerate (constant within stratum)')
            continue
        quic_r1_mean = flat_row['quic']['r2_mean']
        quic_r2_mean = depth_row['quic']['r2_mean']
        concat_r1_mean = flat_row['concat']['r2_mean']
        concat_r2_mean = depth_row['concat']['r2_mean']
        print(f"{target_name:>13} | {depth_row['best_spectral']:+9.3f} | "
              f"{quic_r1_mean:+.3f}±{flat_row['quic']['r2_sd']:.3f} "
              f"{quic_r2_mean:+.3f}±{depth_row['quic']['r2_sd']:.3f} "
              f"{quic_r2_mean - quic_r1_mean:+11.3f} | "
              f"{concat_r1_mean:+.3f}±{flat_row['concat']['r2_sd']:.3f} "
              f"{concat_r2_mean:+.3f}±{depth_row['concat']['r2_sd']:.3f} "
              f"{concat_r2_mean - concat_r1_mean:+11.3f}")

print('\n=== aggregate (equal-weight mean over strata, non-degenerate rows) ===')
print(f'{"target":>13} | {"best_spec":>9} | {"quic_r1":>9} {"quic_r2":>9} '
      f'{"d":>7} | {"cat_r1":>9} {"cat_r2":>9} {"d":>7} | {"strata":>6}')
for target_name in TARGET_NAMES:
    paired_rows = [(RESULTS[stratum_name][target_name],
                    flat_analysis['results'][stratum_name][target_name])
                   for stratum_name in RUN_STRATA
                   if not RESULTS[stratum_name][target_name]['degenerate']]
    if not paired_rows:
        print(f'{target_name:>13} | degenerate in every stratum')
        continue
    aggregate = {}
    for label, extractor in (   # extractors read (depth_row, flat_row)
            ('best_spec', lambda depth, flat: depth['best_spectral']),
            ('quic_r1', lambda depth, flat: flat['quic']['r2_mean']),
            ('quic_r2', lambda depth, flat: depth['quic']['r2_mean']),
            ('cat_r1', lambda depth, flat: flat['concat']['r2_mean']),
            ('cat_r2', lambda depth, flat: depth['concat']['r2_mean'])):
        aggregate[label] = float(np.nanmean(
            [extractor(depth_row, flat_row)
             for depth_row, flat_row in paired_rows]))
    valid_strata_count = sum(
        1 for depth_row, flat_row in paired_rows
        if not np.isnan(depth_row['quic']['r2_mean']))
    print(f"{target_name:>13} | {aggregate['best_spec']:+9.3f} | "
          f"{aggregate['quic_r1']:+9.3f} {aggregate['quic_r2']:+9.3f} "
          f"{aggregate['quic_r2'] - aggregate['quic_r1']:+7.3f} | "
          f"{aggregate['cat_r1']:+9.3f} {aggregate['cat_r2']:+9.3f} "
          f"{aggregate['cat_r2'] - aggregate['cat_r1']:+7.3f} "
          f"| {valid_strata_count}/{len(RUN_STRATA)}")


=== S1_near_regular — reps 1 vs reps 2 (flat encoder) ===
       target | best_spec |       quic_r1       quic_r2    d(r2-r1) |        cat_r1        cat_r2    d(r2-r1)
           C3 |    +1.000 | +0.449±0.714 +0.729±0.176      +0.280 | +0.989±0.002 +0.989±0.002      -0.000
           C4 |    +1.000 | +0.095±0.048 +0.053±0.105      -0.042 | +0.967±0.006 +0.967±0.005      -0.000
           C5 |    +0.802 | -0.017±0.056 +0.029±0.068      +0.045 | +0.863±0.045 +0.776±0.120      -0.087
           C6 |    +0.757 | -0.021±0.012 -0.032±0.034      -0.011 | +0.661±0.058 +0.657±0.056      -0.004
        girth |    +0.468 | +0.007±0.228 -0.093±0.469      -0.099 | +0.436±0.063 +0.417±0.069      -0.019
     diameter |    +0.372 | -0.093±0.122 -0.007±0.007      +0.086 | +0.257±0.146 +0.339±0.057      +0.082
       radius |    +0.028 | -0.064±0.077 -0.020±0.010      +0.044 | -0.033±0.018 -0.041±0.026      -0.008
       wiener |    +0.841 | +0.184±0.147 -0.359±1.080      -0.543 | +0.754±0.083 +0.785±0

## Results

(Written after the run. The reading order: (1) the spot checks, the split
asserts, and the shared-column asserts, which passing silently is what
makes the two arms comparable at all; (2) the three pre-registered
sub-questions against their cells — the C5 live row, the max-degree
gradient under depth, and the S1/S2 C5 concat cells; (3) the aggregate
delta columns, read with the guards from the flat run's closing analysis
intact: linear inaccessibility is what is measured, within-stratum only,
and whichever direction the deltas point is the finding — with E6D's
negative on record, a null here closes the ablation triple (encoder flat,
encoder degree, depth) and the scope claim stands on three measured arms.
Verb discipline as everywhere: the depth arms "differ" or "do not differ"
in linear decodability on these strata; neither arm "learns," "predicts,"
or "generalizes.")